
# Standardisierung von ESM-2 Embeddings (esm2_t33_650M_UR50D-Model)
Um Standardnormalrauschen auf die ESM2-Embeddings zu geben, 
müssen diese auf die Größenordnung des Rauschens ($\mu$ = 0 und $\sigma$ = 1) angepasst werden.
Da ESM-Embeddings über die Dimensionen in Größenordnung schwanken, wird standardiesieruung pro Embedding-Dimension
angewendet, sodass die Als Ausgabe ein tensor der Shape (1280, 2) zurückgegeben wird

In [1]:
# %%Importe
import pandas as pd
import torch
from aguirre.config import DATA_DIR, PROJECT_ROOT, Config
from aguirre.data import read_fasta
from aguirre.esm_wrapper import ESMWrapper

from tqdm.auto import tqdm # Muss nachher tqdm sein
cfg = Config()

In [2]:

# %% Daten zusammenbringen
TRACKED_PATH = PROJECT_ROOT / "wrath_of_god" / "data_tracked" / "split_t075.csv"
DATA_PATH = DATA_DIR / "all19K_unclustered.faa"

df_split = pd.read_csv(filepath_or_buffer= TRACKED_PATH, sep= ",", names= ["ID", "Cluster", "Split"], header = 0)
df_split = df_split[["ID", "Split"]].copy()
maske_0 = df_split["Split"] == "train"
df_split = df_split[maske_0]



In [3]:

ids, seqs = read_fasta(DATA_PATH)
df_AMPs = pd.DataFrame({"ID": ids, "Sequenz": seqs})
# Maskierung mit AMP_IDs im Train-Set
df_AMPs = df_AMPs[df_AMPs["ID"].isin(df_split["ID"])].copy()
AMPs = df_AMPs["Sequenz"].tolist()

print(AMPs)

['FFKKWPWWPWRRK', 'PKTLRKFFARIRGGRAAVLNALGKEEQIGRASNSGRKCARKKK', 'SNTALRRYNQWATGHFM', 'RCICVLGVC', 'TRAAVVLIVIRR', 'IRRWKRVWW', 'LEGEVNKIALLSTNKAVVSLSNGVSVLTSKVLD', 'FLPLVRVAAKLIPSVVCAISKRC', 'FAKKFAKKFKKFAKKFAKFAFAF', 'KYYGNGVTCGKHSCSVDWGKATTCIINNGADAWATGGHQGNHKC', 'SMLAYVDKNDHINPPHSPRS', 'QRFSQPTFKLPQGRLTLSRKF', 'GRPNWGFENDWSCVRVC', 'VRLRIRVAVIRR', 'QARQLVSGLVQQQNNILRALEATQHAVQALVWGVRQLQARVLALERYIK', 'RSGGYCNGKRVCVCR', 'PPISLERLDVGTNLGNAIAKLEAKELLESSDQIL', 'CRRLCYKQRCVTYC', 'RCICGRRFC', 'DISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYE', 'RLVLRLLLGY', 'GFGSLGKALRLGANVL', 'KKVSKKFMKKGSKKVSKKFMKK', 'SGSWLWDVWDWVLHVLSDFKTCLKAKF', 'LKELITTGGQKIGEKIRRIGQRIKDFFKNLQPREEKS', 'RCLCRRGFC', 'SDDPKESEGDLHCVCVKTTSKV', 'RSVIFGCTKSIPPICFVGFK', 'KSCCKDTTARNCYNVCRIPGTPRPVCATTCRCKIISGNKCPKDYPK', 'GLFTLIKGAAKLIGKTTAKEAGKTGKLEMACKITNQC', 'GLPVCGETCVGGTCNTPGCGCSWPVCTRN', 'APKRKSGVSKC', 'KFHEKHHSHRGY', 'KKCRRWQWRMKKLGA', 'GHRRYFTFGAGYVYF', 'RWRRLLKKLHHLLHGGGFKKFWKWFRRF', 'GFCFRVCAKRNGVRKCYRRCN', 'RWWKIWVIRWRR', '

In [4]:
#Pfad zum Speichern 
STANDARS_PATH = PROJECT_ROOT / "wrath_of_god" / "data_tracked" / "standards_650M.csv"
B = 16 #batchsize 8M : 256, 650M:16
AMPs_sortiert = sorted(AMPs, key=len)
start_positionen = list(range(0, len(AMPs_sortiert), B)) #Sortieren führt dazu, dass Sequenzen gleicher Länge zusammenkommen (encode dimension für alle Embeddings abhängig von längster Sequenz)
batches = [AMPs_sortiert[i:i+B] for i in start_positionen]
print(f"{len(AMPs)} Sequenzen -> {len(batches)} Batches")
print(f"erster Batch: {len(batches[0])}, letzter: {len(batches[-1])}")


15507 Sequenzen -> 970 Batches
erster Batch: 16, letzter: 3


In [5]:
#Batch-Schleife
x_1 = torch.zeros(cfg.embed_dim, dtype= torch.float64)
x_2 = x_1.clone()
n = 0
wrapper = ESMWrapper(cfg = cfg)

for b in batches: 
    EMBs, lengths = wrapper.encode(b) # Shape (B,L_max, D)
    summe_1 = EMBs.sum(dim = (0,1))
    summe_2 = EMBs**2
    summe_2 = summe_2.sum(dim = (0,1))
    x_1 += summe_1
    x_2 += summe_2
    n += sum(lengths)

mu = x_1 / n
sigma = (x_2 / n - mu**2).clamp_min(0).sqrt()
dict = {
    "mean": mu.detach().cpu().numpy(),
    "std": sigma.detach().cpu().numpy()
}
df_standards = pd.DataFrame(dict)
df_standards.to_csv(STANDARS_PATH, index_label= "dim")

#Prüfungen: 

print(f"n = {n}   (erwartet: 331383)")
assert n == 331383, f"Residuenzahl weicht ab: {n}"
assert (sigma > 0).all(), f"{(sigma == 0).sum().item()} Dimensionen mit sigma = 0"
assert torch.isfinite(mu).all() and torch.isfinite(sigma).all()

# Gegenprobe: einen Batch normalisieren und nachmessen
EMBs, lengths = wrapper.encode(batches[len(batches) // 2])
x = EMBs.double()
maske = torch.arange(x.shape[1])[None, :] < torch.tensor(lengths)[:, None]   # (B, L_max)
echte = x[maske]                                                             # (N_echt, D)
z = (echte - mu) / sigma
print(f"nach Normalisierung:  Mittel = {z.mean():+.4f}   Std = {z.std():.4f}")





n = 331383   (erwartet: 331383)
nach Normalisierung:  Mittel = +0.0002   Std = 0.9229


In [6]:
import random
random.seed(0)
probe = random.sample(AMPs, 128)

EMBs, lengths = wrapper.encode(probe)
x = EMBs.double()
maske = torch.arange(x.shape[1])[None, :] < torch.tensor(lengths)[:, None]
z = (x[maske] - mu) / sigma
print(f"Mittel = {z.mean():+.4f}   Std = {z.std():.4f}")

Mittel = +0.0001   Std = 0.9892


In [ ]:
print(df_split.shape)
print(df_AMPs.head(10), df_AMPs.shape)
print(df_split.head(10))

